In [ ]:
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS_DIR = Path("../data/experiment_results")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METHOD_DISPLAY = {
    "bide_coverage": "BIDE+Coverage",
    "frequency_vector": "Freq. Vector",
    "ngram": "N-gram",
    "taspm": "TaSPM",
    "process_conformance": "Conformance",
    "deeplog": "DeepLog",
    "bilstm": "Bi-LSTM",
    "step_count": "Step Count",
}

palette = sns.color_palette("tab10", n_colors=len(METHOD_DISPLAY))
METHOD_COLORS = {name: palette[i] for i, name in enumerate(METHOD_DISPLAY)}

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "figure.figsize": (10, 6),
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.1,
})
sns.set_style("whitegrid")


def save_fig(fig, name):
    """Save figure as PNG and PDF."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}")


def pretty_method(key):
    return METHOD_DISPLAY.get(key, key)


print("Setup complete.")

## Section 1: Load All Experiments

In [ ]:
CONFIG_DEFAULTS = {
    "balanced": False,
    "exclude_errors": False,
    "exclude_timeouts": False,
    "ngram_ns": "10",
}


def derive_experiment_name(config, folder_name):
    """Build a readable experiment name from config flags."""
    parts = []
    if config.get("exclude_errors", False):
        parts.append("excl-err")
    if config.get("exclude_timeouts", False):
        parts.append("excl-to")
    if config.get("balanced", False):
        parts.append("balanced")
    ngram = config.get("ngram_ns", "10")
    if ngram != "10":
        parts.append(f"ngram={ngram}")
    if not parts:
        parts.append("default")
    return "+".join(parts)


rows = []
experiment_configs = {}

for folder in sorted(RESULTS_DIR.iterdir()):
    if not folder.is_dir() or folder.is_symlink():
        continue

    results_path = folder / "results.json"
    config_path = folder / "config.json"
    if not results_path.exists():
        continue

    with open(results_path) as f:
        results = json.load(f)

    config = {}
    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)

    folder_name = folder.name
    try:
        ts = datetime.strptime(folder_name[:19], "%Y-%m-%d_%H-%M-%S")
    except ValueError:
        ts = datetime.strptime(folder_name[:19], "%Y-%m-%d_%H_%M_%S")

    exp_name = derive_experiment_name(config, folder_name)
    experiment_configs[folder_name] = {"config": config, "name": exp_name, "timestamp": ts}

    for baseline, bdata in results.get("per_baseline", {}).items():
        for k_str, metrics in bdata.get("at_k", {}).items():
            k_val = int(k_str)
            threshold = bdata.get("thresholds", {}).get(k_str, np.nan)
            row = {
                "timestamp": ts,
                "folder": folder_name,
                "experiment_name": exp_name,
                "baseline": baseline,
                "k": k_val,
                "threshold": threshold,
                "failure_rate": results.get("failure_rate", np.nan),
                **{m: metrics.get(m, np.nan) for m in
                   ["precision", "recall", "f1", "f1_success", "accuracy", "auc_roc", "auc_pr"]},
                **{param: config.get(param, default)
                   for param, default in CONFIG_DEFAULTS.items()},
            }
            rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values(["timestamp", "baseline", "k"]).reset_index(drop=True)

n_experiments = df["folder"].nunique()
date_range = f"{df['timestamp'].min():%Y-%m-%d} to {df['timestamp'].max():%Y-%m-%d}"
unique_baselines = sorted(df["baseline"].unique())
unique_configs = df["experiment_name"].unique().tolist()

print(f"Loaded {n_experiments} experiments ({date_range})")
print(f"Unique baselines ({len(unique_baselines)}): {', '.join(unique_baselines)}")
print(f"Unique configs: {', '.join(unique_configs)}")
print(f"\nDataFrame shape: {df.shape}")
df.head(10)

## Section 2: Cross-Experiment Comparison Table

In [ ]:
K_SELECT = 10

df_k = df[df["k"] == K_SELECT].copy()

pivot = df_k.pivot_table(
    index="baseline",
    columns=["timestamp", "experiment_name"],
    values="auc_pr",
    aggfunc="first",
)
pivot.columns = [f"{ts:%m-%d} {name}" for ts, name in pivot.columns]
pivot.index = pivot.index.map(pretty_method)


def highlight_max(s):
    is_max = s == s.max()
    return ["font-weight: bold; background-color: #d4edda" if v else "" for v in is_max]


styled = (
    pivot.style
    .apply(highlight_max, axis=0)
    .format("{:.4f}", na_rep="—")
    .set_caption(f"AUC-PR at K={K_SELECT} across experiments (best per experiment highlighted)")
)
display(styled)

print("\n--- Config parameters that vary between experiments ---")
config_df = (
    df_k.drop_duplicates("folder")
    [["experiment_name", "balanced", "exclude_errors", "exclude_timeouts", "ngram_ns"]]
    .set_index("experiment_name")
)
varying = [c for c in config_df.columns if config_df[c].nunique() > 1]
display(config_df[varying] if varying else "All config parameters are identical.")

## Section 3: Method Performance Over Time

In [ ]:
df_time = df_k.copy()
df_time["method"] = df_time["baseline"].map(pretty_method)

fig, ax = plt.subplots(figsize=(12, 6))
for baseline in sorted(df_time["baseline"].unique()):
    sub = df_time[df_time["baseline"] == baseline].sort_values("timestamp")
    ax.plot(
        sub["timestamp"], sub["auc_pr"],
        marker="o", markersize=5, linewidth=1.5,
        label=pretty_method(baseline),
        color=METHOD_COLORS.get(baseline),
    )

ax.set_xlabel("Experiment Date")
ax.set_ylabel("AUC-PR")
ax.set_title(f"Method Performance Over Time (K={K_SELECT})")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
fig.autofmt_xdate(rotation=30)
fig.tight_layout()
save_fig(fig, "performance_over_time")
plt.show()

## Section 4: Detailed Comparison for Latest Run

In [ ]:
latest_ts = df["timestamp"].max()
df_latest = df[(df["timestamp"] == latest_ts) & (df["k"] == K_SELECT)].copy()
df_latest["method"] = df_latest["baseline"].map(pretty_method)

metrics_to_show = ["auc_pr", "f1", "f1_success", "precision", "recall"]
metric_labels = ["AUC-PR", "F1", "F1 (success)", "Precision", "Recall"]

methods = df_latest.sort_values("auc_pr", ascending=False)["baseline"].tolist()
method_labels = [pretty_method(m) for m in methods]

x = np.arange(len(methods))
width = 0.15
offsets = np.arange(len(metrics_to_show)) - (len(metrics_to_show) - 1) / 2

metric_colors = sns.color_palette("Set2", n_colors=len(metrics_to_show))

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, label) in enumerate(zip(metrics_to_show, metric_labels)):
    values = [df_latest[df_latest["baseline"] == m][metric].values[0] for m in methods]
    ax.bar(x + offsets[i] * width, values, width, label=label, color=metric_colors[i])

ax.set_xticks(x)
ax.set_xticklabels(method_labels, rotation=25, ha="right")
ax.set_ylabel("Score")
ax.set_title(f"Latest Experiment — All Metrics at K={K_SELECT}")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
fig.tight_layout()
save_fig(fig, "latest_comparison")
plt.show()

## Section 5: Imbalanced vs Balanced Comparison

In [ ]:
df_bal = df[(df["k"] == K_SELECT) & (df["balanced"] == True)]
df_imbal = df[(df["k"] == K_SELECT) & (df["balanced"] == False)]

if df_bal.empty:
    print("No balanced experiments found — skipping section.")
else:
    latest_bal_ts = df_bal["timestamp"].max()
    latest_imbal_ts = df_imbal["timestamp"].max()

    bal = df_bal[df_bal["timestamp"] == latest_bal_ts].set_index("baseline")
    imbal = df_imbal[df_imbal["timestamp"] == latest_imbal_ts].set_index("baseline")

    all_baselines = sorted(set(bal.index) | set(imbal.index))

    comparison = pd.DataFrame({
        "Imbalanced AUC-PR": imbal["auc_pr"].reindex(all_baselines),
        "Balanced AUC-PR": bal["auc_pr"].reindex(all_baselines),
        "Imbalanced F1": imbal["f1"].reindex(all_baselines),
        "Balanced F1": bal["f1"].reindex(all_baselines),
    })
    comparison.index = comparison.index.map(pretty_method)
    display(comparison.style.format("{:.4f}", na_rep="—")
            .set_caption("Imbalanced vs Balanced — Most Recent of Each"))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax_i, (metric, title) in enumerate([("auc_pr", "AUC-PR"), ("f1", "F1")]):
        x = np.arange(len(all_baselines))
        w = 0.35
        vals_imbal = [imbal[metric].get(b, np.nan) for b in all_baselines]
        vals_bal = [bal[metric].get(b, np.nan) for b in all_baselines]
        axes[ax_i].bar(x - w / 2, vals_imbal, w, label="Imbalanced", color="steelblue")
        axes[ax_i].bar(x + w / 2, vals_bal, w, label="Balanced", color="coral")
        axes[ax_i].set_xticks(x)
        axes[ax_i].set_xticklabels([pretty_method(b) for b in all_baselines], rotation=30, ha="right")
        axes[ax_i].set_title(title)
        axes[ax_i].legend()
        axes[ax_i].set_ylim(0, 1.05)

    fig.suptitle(f"Imbalanced vs Balanced at K={K_SELECT}", fontsize=14, y=1.02)
    fig.tight_layout()
    save_fig(fig, "balanced_comparison")
    plt.show()

## Section 6: N-gram Ablation

In [ ]:
ngram_configs = df["ngram_ns"].unique()

if len(ngram_configs) <= 1:
    print(f"Only one ngram_ns config found ({ngram_configs}) — skipping ablation.")
else:
    df_ngram = df[(df["k"] == K_SELECT) & (df["baseline"] == "ngram")].copy()
    df_ngram = df_ngram.sort_values("ngram_ns")

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(
        range(len(df_ngram)),
        df_ngram["auc_pr"].values,
        color=sns.color_palette("viridis", n_colors=len(df_ngram)),
        edgecolor="black", linewidth=0.5,
    )
    ax.set_xticks(range(len(df_ngram)))
    ax.set_xticklabels([f"n={v}" for v in df_ngram["ngram_ns"].values], rotation=0)
    ax.set_ylabel("AUC-PR")
    ax.set_title(f"N-gram Ablation — AUC-PR at K={K_SELECT}")
    ax.set_ylim(0, 1.05)

    for bar, val in zip(bars, df_ngram["auc_pr"].values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)

    fig.tight_layout()
    save_fig(fig, "ngram_ablation")
    plt.show()

## Section 7: Experiment Variant Comparison

In [ ]:
variant_defs = {
    "excl-err": {"exclude_errors": True, "balanced": False, "exclude_timeouts": False},
    "excl-err+balanced": {"exclude_errors": True, "balanced": True, "exclude_timeouts": False},
    "excl-err+excl-to": {"exclude_errors": True, "balanced": False, "exclude_timeouts": True},
}

variant_data = {}
for vname, vfilter in variant_defs.items():
    mask = df["k"] == K_SELECT
    for param, val in vfilter.items():
        mask = mask & (df[param] == val)
    matched = df[mask]
    if not matched.empty:
        latest = matched[matched["timestamp"] == matched["timestamp"].max()]
        variant_data[vname] = latest.set_index("baseline")["auc_pr"]

if len(variant_data) < 2:
    print(f"Found only {len(variant_data)} variant(s) — need at least 2 for comparison. Skipping.")
else:
    all_baselines = sorted(set().union(*[v.index for v in variant_data.values()]))
    variant_names = list(variant_data.keys())

    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(all_baselines))
    n_variants = len(variant_names)
    width = 0.8 / n_variants
    variant_colors = sns.color_palette("husl", n_colors=n_variants)

    for i, (vname, series) in enumerate(variant_data.items()):
        offset = (i - (n_variants - 1) / 2) * width
        vals = [series.get(b, np.nan) for b in all_baselines]
        ax.bar(x + offset, vals, width, label=vname, color=variant_colors[i], edgecolor="black", linewidth=0.3)

    ax.set_xticks(x)
    ax.set_xticklabels([pretty_method(b) for b in all_baselines], rotation=30, ha="right")
    ax.set_ylabel("AUC-PR")
    ax.set_title(f"Experiment Variant Comparison at K={K_SELECT}")
    ax.set_ylim(0, 1.05)
    ax.legend(title="Variant")
    fig.tight_layout()
    save_fig(fig, "variant_comparison")
    plt.show()

## Section 8: Per-K Performance Curves

In [ ]:
df_latest_all_k = df[df["timestamp"] == latest_ts].copy()

metrics_per_k = [("auc_pr", "AUC-PR"), ("f1", "F1"), ("f1_success", "F1 (success)")]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, (metric, label) in zip(axes, metrics_per_k):
    for baseline in sorted(df_latest_all_k["baseline"].unique()):
        sub = df_latest_all_k[df_latest_all_k["baseline"] == baseline].sort_values("k")
        ax.plot(
            sub["k"], sub[metric],
            marker="o", markersize=6, linewidth=1.8,
            label=pretty_method(baseline),
            color=METHOD_COLORS.get(baseline),
        )
    ax.set_xlabel("K (prefix length)")
    ax.set_ylabel(label)
    ax.set_title(f"{label} vs K")
    ax.set_xticks(sorted(df_latest_all_k["k"].unique()))
    ax.set_ylim(0, 1.05)

axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

fig.suptitle("Per-K Performance Curves (Latest Experiment)", fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, "per_k_curves")
plt.show()

## Section 9: Summary of Key Findings

In [ ]:
from IPython.display import Markdown, display as ipy_display

latest_k10 = df[(df["timestamp"] == latest_ts) & (df["k"] == K_SELECT)].set_index("baseline")
best_aucpr = latest_k10["auc_pr"].idxmax()
best_aucpr_val = latest_k10.loc[best_aucpr, "auc_pr"]
best_f1s = latest_k10["f1_success"].idxmax()
best_f1s_val = latest_k10.loc[best_f1s, "f1_success"]

all_k10 = df[df["k"] == K_SELECT]
overall_best_row = all_k10.loc[all_k10["auc_pr"].idxmax()]

bal_exists = not df[(df["k"] == K_SELECT) & (df["balanced"] == True)].empty
if bal_exists:
    bal_best = df[(df["k"] == K_SELECT) & (df["balanced"] == True)].sort_values("auc_pr", ascending=False).iloc[0]
    imbal_match = df[(df["k"] == K_SELECT) & (df["balanced"] == False) & (df["baseline"] == bal_best["baseline"])]
    if not imbal_match.empty:
        imbal_best = imbal_match.sort_values("auc_pr", ascending=False).iloc[0]
        bal_summary = (
            f"For {pretty_method(bal_best['baseline'])}, balancing "
            f"{'improved' if bal_best['auc_pr'] > imbal_best['auc_pr'] else 'decreased'} "
            f"AUC-PR from {imbal_best['auc_pr']:.4f} to {bal_best['auc_pr']:.4f}."
        )
    else:
        bal_summary = f"Best balanced result: {pretty_method(bal_best['baseline'])} AUC-PR={bal_best['auc_pr']:.4f}."
else:
    bal_summary = "No balanced experiments were found."

ngram_configs_list = sorted(df["ngram_ns"].unique())
if len(ngram_configs_list) > 1:
    ngram_rows = df[(df["k"] == K_SELECT) & (df["baseline"] == "ngram")]
    best_ng = ngram_rows.loc[ngram_rows["auc_pr"].idxmax()]
    ngram_summary = (
        f"Across n-gram configurations ({', '.join(str(c) for c in ngram_configs_list)}), "
        f"the best AUC-PR was {best_ng['auc_pr']:.4f} with ngram_ns={best_ng['ngram_ns']}."
    )
else:
    ngram_summary = f"Only one n-gram configuration tested (ngram_ns={ngram_configs_list[0]})."

ee_data = df[(df["k"] == K_SELECT) & (df["exclude_errors"] == True) & (df["balanced"] == False) & (df["exclude_timeouts"] == False)]
variant_summary = ""
if not ee_data.empty:
    ee_best = ee_data.loc[ee_data["auc_pr"].idxmax()]
    default_match = all_k10[
        (all_k10["baseline"] == ee_best["baseline"]) &
        (all_k10["exclude_errors"] == False) & (all_k10["balanced"] == False)
    ]
    if not default_match.empty:
        def_val = default_match.sort_values("auc_pr", ascending=False).iloc[0]["auc_pr"]
        delta = ee_best["auc_pr"] - def_val
        variant_summary = (
            f"Excluding error traces {'improved' if delta > 0 else 'decreased'} "
            f"the best method's AUC-PR by {abs(delta):.4f} "
            f"({def_val:.4f} -> {ee_best['auc_pr']:.4f} for {pretty_method(ee_best['baseline'])})."
        )

k3 = df[(df["timestamp"] == latest_ts) & (df["k"] == 3) & (df["baseline"] == best_aucpr)]
k3_val = k3["auc_pr"].values[0] if not k3.empty else np.nan
k_trend = (
    f"For {pretty_method(best_aucpr)}, AUC-PR improves from {k3_val:.4f} at K=3 "
    f"to {best_aucpr_val:.4f} at K={K_SELECT}, indicating longer prefixes help discrimination."
) if not np.isnan(k3_val) else ""

summary_md = f"""\
### Key Findings

**Overall Best Method (K={K_SELECT}):** {pretty_method(best_aucpr)} achieves the highest AUC-PR \
of {best_aucpr_val:.4f}. The best F1 on successful traces belongs to \
{pretty_method(best_f1s)} ({best_f1s_val:.4f}).

**Cross-Experiment:** The globally best AUC-PR observed across all experiments is \
{overall_best_row['auc_pr']:.4f} ({pretty_method(overall_best_row['baseline'])}, \
config: {overall_best_row['experiment_name']}).

**Imbalanced vs Balanced:** {bal_summary}

**N-gram Ablation:** {ngram_summary}

**Variant Comparison:** {variant_summary if variant_summary else 'No exclude-errors variant data available for comparison.'}

**Per-K Trends:** {k_trend if k_trend else 'Insufficient data to compare K values.'}

**Failure Rate:** The latest experiment has a failure rate of \
{latest_k10['failure_rate'].iloc[0]:.1%}, indicating a heavily imbalanced dataset \
which motivates the use of AUC-PR over accuracy as the primary metric.
"""

ipy_display(Markdown(summary_md))